# Notebook 03 — Predictive Geospatial Models
Three spatial models: DBSCAN seller clustering, GWR delivery time, Moran's I satisfaction.

In [7]:
import src.models
import src.geospatial_utils
import src.data_loader
from sklearn.preprocessing import StandardScaler
import pathlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from src.models import (
    run_dbscan, build_folium_cluster_map,
    run_gwr, build_gwr_coefficient_map,
    compute_morans_i, build_lisa_map,
)
from src.geospatial_utils import add_seller_buyer_distance, to_geodataframe
import importlib
import sys
sys.path.insert(0, '..')

importlib.reload(src.data_loader)
importlib.reload(src.geospatial_utils)
importlib.reload(src.models)


OUTPUTS = pathlib.Path('..') / 'outputs'
PROC = OUTPUTS / 'processed_data'
print('Imports OK')

Imports OK


In [8]:
centroids = pd.read_csv(PROC / 'geo_centroids.csv')
sellers = pd.read_csv(PROC / 'olist_sellers_dataset.csv')
orders = pd.read_csv(PROC / 'olist_orders_dataset.csv')
items = pd.read_csv(PROC / 'olist_order_items_dataset.csv')
customers = pd.read_csv(PROC / 'olist_customers_dataset.csv')
reviews = pd.read_csv(PROC / 'olist_order_reviews_dataset.csv')

# Renamed once here so both GWR and LISA sections can use it without re-defining
centroids_renamed = centroids.rename(
    columns={'geolocation_zip_code_prefix': 'zip'})

print('All tables loaded.')

All tables loaded.


## A) DBSCAN — Seller Clustering

In [9]:
# Join sellers with centroid coordinates
sellers_geo = sellers.merge(
    centroids.rename(
        columns={'geolocation_zip_code_prefix': 'seller_zip_code_prefix'}),
    on='seller_zip_code_prefix',
    how='inner'
)
print(f'Sellers with coordinates: {len(sellers_geo):,}')
display(sellers_geo.head(3))

Sellers with coordinates: 3,088


,seller_id,seller_zip_code_prefix,seller_city,seller_state,lat,lng
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,-22.893848,-47.061337
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,-22.383437,-46.947927
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,-22.909572,-43.177703


In [10]:
coords = sellers_geo[['lat', 'lng']].values
labels = run_dbscan(coords, eps_km=50, min_samples=5)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise = (labels == -1).sum()
print(f'Clusters found: {n_clusters}  |  Noise points: {noise:,}')

Clusters found: 26  |  Noise points: 149


In [11]:
sellers_geo['cluster'] = labels
summary = (
    sellers_geo[sellers_geo['cluster'] >= 0]
    .groupby('cluster')
    .agg(size=('seller_id', 'count'), center_lat=('lat', 'mean'), center_lng=('lng', 'mean'))
    .sort_values('size', ascending=False)
)
display(summary.head(10))

,size,center_lat,center_lng
cluster,,,
0,2117,-22.982198,-47.031705
3,176,-27.154497,-48.928439
1,170,-25.462418,-49.267848
6,119,-19.944969,-44.127342
7,101,-29.665816,-51.248481
14,41,-24.952775,-53.888666
5,30,-15.829420,-47.970239
2,28,-16.653282,-49.256054
16,17,-20.308831,-40.341292


In [ ]:
from IPython.display import IFrame
m_clusters = build_folium_cluster_map(sellers_geo, labels)

sellers_geo[['seller_id', 'seller_zip_code_prefix', 'seller_state', 'lat', 'lng', 'cluster']].to_csv(
    PROC / 'seller_clusters.csv', index=False
)
print('seller_clusters.csv saved.')

IFrame(src='../outputs/maps/seller_clusters.html', width='100%', height='500px')

Cluster map saved → C:\Users\Angie\OneDrive\Documentos\Analista de Datos\PORTAFOLIO\PROYECTO_3\outputs\maps\seller_clusters.html
seller_clusters.csv saved.


## B) GWR — Delivery Time

In [13]:
# Build enriched orders DataFrame
# centroids_renamed is defined in cell-load-data

# Seller coordinates
from src.geospatial_utils import haversine_distance
seller_coords = sellers.merge(
    centroids_renamed, left_on='seller_zip_code_prefix', right_on='zip', how='inner'
)[['seller_id', 'seller_state', 'lat', 'lng']].rename(columns={'lat': 'seller_lat', 'lng': 'seller_lng'})

# Customer coordinates
customer_coords = customers.merge(
    centroids_renamed, left_on='customer_zip_code_prefix', right_on='zip', how='inner'
)[['customer_id', 'customer_state', 'lat', 'lng']].rename(columns={'lat': 'customer_lat', 'lng': 'customer_lng'})

# Order items — one row per order
items_1 = items[['order_id', 'seller_id', 'price']].drop_duplicates('order_id')

# Join everything
gwr_df = (
    orders[['order_id', 'customer_id', 'order_purchase_timestamp',
            'order_delivered_customer_date', 'order_status']]
    .query('order_status == "delivered"')
    .merge(items_1, on='order_id', how='left')
    .merge(seller_coords, on='seller_id', how='inner')
    .merge(customer_coords, on='customer_id', how='inner')
)

# Compute delivery_days and distance_km
gwr_df['order_purchase_timestamp'] = pd.to_datetime(
    gwr_df['order_purchase_timestamp'])
gwr_df['order_delivered_customer_date'] = pd.to_datetime(
    gwr_df['order_delivered_customer_date'])
gwr_df['delivery_days'] = (
    gwr_df['order_delivered_customer_date'] - gwr_df['order_purchase_timestamp']).dt.days

gwr_df['distance_km'] = haversine_distance(
    gwr_df['seller_lat'].values, gwr_df['seller_lng'].values,
    gwr_df['customer_lat'].values, gwr_df['customer_lng'].values
)

gwr_df = gwr_df.dropna(subset=['delivery_days', 'distance_km'])
gwr_df = gwr_df[(gwr_df['delivery_days'] > 0) &
                (gwr_df['delivery_days'] < 120)]
print(f'GWR dataset rows: {len(gwr_df):,}')

GWR dataset rows: 95,935


In [14]:
# Subsample for speed (GWR is O(n^2))
SAMPLE = 2000
gwr_sample = gwr_df.sample(n=min(SAMPLE, len(gwr_df)), random_state=42).copy()

scaler = StandardScaler()
gwr_sample['distance_km_scaled'] = scaler.fit_transform(
    gwr_sample[['distance_km']])

y = gwr_sample['delivery_days'].values.astype(float)
# mgwr requires an explicit intercept column as the first column of X
X = np.column_stack([np.ones(len(gwr_sample)),
                    gwr_sample['distance_km_scaled'].values]).astype(float)
coords = gwr_sample[['seller_lng', 'seller_lat']].values.astype(float)

print(f'y: {y.shape}  X: {X.shape}  coords: {coords.shape}')
print('X[:,0]=intercept  X[:,1]=distance_km_scaled')

y: (2000,)  X: (2000, 2)  coords: (2000, 2)
X[:,0]=intercept  X[:,1]=distance_km_scaled


In [15]:
gwr_results = run_gwr(y, X, coords)
print(f'GWR R2: {gwr_results.R2:.4f}')

Optimal bandwidth: 289.0
GWR R2: 0.2140


In [ ]:
from IPython.display import IFrame
m_gwr = build_gwr_coefficient_map(gwr_sample, gwr_results, predictor_idx=1)
print(f'GWR R2: {gwr_results.R2:.4f}')

IFrame(src='../outputs/maps/gwr_coefficients.html',
       width='100%', height='500px')

GWR coefficient map saved → C:\Users\Angie\OneDrive\Documentos\Analista de Datos\PORTAFOLIO\PROYECTO_3\outputs\maps\gwr_coefficients.html
GWR R2: 0.2140


## C) Moran's I — Customer Satisfaction by State

In [17]:
# Aggregate avg review score per state
review_orders = orders[['order_id', 'customer_id']].merge(
    customers[['customer_id', 'customer_state']], on='customer_id'
).merge(
    reviews[['order_id', 'review_score']], on='order_id'
)
state_scores = review_orders.groupby('customer_state')[
    'review_score'].mean().reset_index()
state_scores.columns = ['state', 'avg_review_score']
print(f'States with scores: {len(state_scores)}')
display(state_scores.sort_values('avg_review_score'))

States with scores: 27


,state,avg_review_score
21,RR,3.608696
1,AL,3.751208
9,MA,3.764075
24,SE,3.808023
13,PA,3.849174
5,CE,3.851016
4,BA,3.860888
18,RJ,3.874971
16,PI,3.920570
15,PE,4.011543


In [18]:
# Build state-level GeoDataFrame using seller centroids (one per state)
from src.geospatial_utils import to_geodataframe
state_centroids = (
    sellers.merge(centroids_renamed, left_on='seller_zip_code_prefix',
                  right_on='zip', how='inner')
    .groupby('seller_state')[['lat', 'lng']].mean().reset_index()
    .rename(columns={'seller_state': 'state'})
)

state_gdf_df = state_scores.merge(state_centroids, on='state', how='inner')
print(f'States with geo: {len(state_gdf_df)}')

state_gdf = to_geodataframe(state_gdf_df, lat_col='lat', lon_col='lng')
state_gdf['name'] = state_gdf['state']
display(state_gdf.head())

States with geo: 23


,state,avg_review_score,lat,lng,geometry,name
0,AC,4.049383,-9.967843,-67.813284,POINT (-67.81328 -9.96784),AC
1,AM,4.183673,-3.131672,-60.019225,POINT (-60.01923 -3.13167),AM
2,BA,3.860888,-13.245681,-39.287298,POINT (-39.2873 -13.24568),BA
3,CE,3.851016,-4.289135,-38.846748,POINT (-38.84675 -4.28914),CE
4,DF,4.064711,-15.813846,-47.968331,POINT (-47.96833 -15.81385),DF


In [19]:
from libpysal.weights import KNN
# Use KNN (k=4) — robust when not all states share a border
coords_arr = np.column_stack([state_gdf.geometry.x, state_gdf.geometry.y])
w = KNN.from_array(coords_arr, k=4)
w.transform = 'r'
print(f'Spatial weights built: {w.n} units')

Spatial weights built: 23 units


In [20]:
values = state_gdf['avg_review_score'].values
moran_global, moran_local = compute_morans_i(values, w)

Moran's I = 0.0872  p-value = 0.1530


In [21]:
from IPython.display import IFrame
m_lisa = build_lisa_map(state_gdf, moran_local, value_col='avg_review_score')
IFrame(src='../outputs/maps/lisa_satisfaction.html',
       width='100%', height='500px')

LISA map saved → C:\Users\Angie\OneDrive\Documentos\Analista de Datos\PORTAFOLIO\PROYECTO_3\outputs\maps\lisa_satisfaction.html
